# Real daily limit-order research

This notebook uses real SPY daily OHLCV data from Twelve Data. It studies one-session buy limits set a fixed percentage below the preceding close. It does not use the monthly Shiller series and cannot produce future-dated rows.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from retail_sp500.daily_data import load_or_fetch_twelve_data_daily
from retail_sp500.limit_orders import evaluate_limit_discount_grid, one_session_limit_outcomes

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

## Load real daily data

Create a free Twelve Data API key and export it as `TWELVE_DATA_API_KEY`. The first run caches the validated CSV under `data/processed/`; later runs use that cache unless `REFRESH = True`.

In [ ]:
SYMBOL = "SPY"
START_DATE = "2007-06-01"
CACHE_PATH = Path("data/processed/spy_daily_1day.csv")
REFRESH = False

daily = load_or_fetch_twelve_data_daily(
    os.getenv("TWELVE_DATA_API_KEY"),
    cache_path=CACHE_PATH,
    refresh=REFRESH,
    symbol=SYMBOL,
    start_date=START_DATE,
)

summary = {
    "source": daily.attrs.get("source"),
    "symbol": daily.attrs.get("symbol", SYMBOL),
    "start": daily.index.min(),
    "end": daily.index.max(),
    "sessions": len(daily),
}
print(summary)
assert daily.index.max() <= pd.Timestamp.today().normalize()
daily.tail()

In [ ]:
px.line(
    daily.reset_index(),
    x="date",
    y="close",
    title=f"{SYMBOL} real daily close",
).show()

## Fixed-discount grid

Each order is set after the previous close and is valid for the next session only. Gap-down fills receive the opening price; an intraday touch fills at the limit; an unfilled order remains cash for that session.

In [ ]:
discounts = np.arange(0.0, 0.0501, 0.001)
grid = evaluate_limit_discount_grid(daily, discounts)

grid.sort_values("mean_one_session_excess_vs_open", ascending=False).head(15)

In [ ]:
px.line(
    grid,
    x="discount",
    y="fill_rate",
    title="Next-session limit fill rate",
    labels={"discount": "Limit below previous close", "fill_rate": "Fill rate"},
).show()

In [ ]:
px.line(
    grid,
    x="discount",
    y="mean_one_session_excess_vs_open",
    title="Mean one-session value difference versus buying at the open",
    labels={
        "discount": "Limit below previous close",
        "mean_one_session_excess_vs_open": "Limit value minus market-at-open value",
    },
).show()

## Candidate under this exact one-session objective

This is not yet a universal portfolio optimum. It is the best historical discount only for the stated next-session experiment and date range.

In [ ]:
candidate = grid.loc[grid["mean_one_session_excess_vs_open"].idxmax()]
candidate

In [ ]:
candidate_outcomes = one_session_limit_outcomes(daily, float(candidate["discount"]))
candidate_outcomes.tail(20)

## Next execution layer

The portfolio-grade model should add persistent contribution lots, multi-day expiry, market-on-expiry rules, dividends, split handling, cash yield and transaction costs. Those choices change the optimal discount, so they must be explicit rather than hidden in the notebook.